# Project 06: Plant Leaf Disease Classification
### Deep Convolutional Neural Networks & Transfer Learning

Classifying crop pathology from RGB leaf imagery to enable early agricultural intervention.

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from dataclasses import dataclass

# Define CNN architecture for plant leaf disease classification
class PlantDiseaseClassifier(nn.Module):
    def __init__(self, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 112x112
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, 4)) # 4x4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
        
    def forward(self, x):
        return self.classifier(self.features(x))

model = PlantDiseaseClassifier(num_classes=4)
print("Model Architecture Summary:")
print(model)

# Test forward pass with batch of 2 RGB images (224x224)
dummy_batch = torch.randn(2, 3, 224, 224)
logits = model(dummy_batch)
probs = torch.softmax(logits, dim=1)
classes = ["Healthy", "Bacterial_Spot", "Early_Blight", "Late_Blight"]
print("\nPredicted Class Probabilities:")
for i, p in enumerate(probs):
    pred_class = classes[torch.argmax(p).item()]
    print(f"Sample {i}: Predicted '{pred_class}' with confidence {torch.max(p).item():.2%}")